# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² open dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata as a dict
metadata_dict = dataset.metadata.to_json()

print('Dataset loaded:')
print(f"Name: {metadata_dict.get('name')}")
print(f"Description: {metadata_dict.get('description')}")


## 2. Data Overview
Review available record sets, fields, and their IDs for this dataset using `mlcroissant`.

Each entity (record set, field, column) is referenced by its `@id`.

In [ ]:
from collections import defaultdict

# List all record set @ids in the dataset
record_sets = dataset.record_sets
print('Available record sets and their field @ids:')

record_set_ids = []
fields_by_record_set = defaultdict(list)

for rs in record_sets:
    print(f"Record set name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    # List fields for each record set
    for field in rs.fields:
        print(f" - Field: {field.name}, @id: {field.id}")
        fields_by_record_set[rs.id].append(field.id)

# Optionally, display first few records of any one record set
if record_set_ids:
    print(f"\nSample records from first record set '@id': {record_set_ids[0]}")
    count = 0
    for rec in dataset.records(record_set=record_set_ids[0]):
        pprint.pprint(rec)
        count += 1
        if count >= 2:
            break
else:
    print('No record sets found in this Croissant package.')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis.

All record sets are identified by their `@id`.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}
for recset_id in record_set_ids:
    # Load all rows for this record set
    rows = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(rows)
    dataframes[recset_id] = df
    print(f"Record set '@id': {recset_id} with shape {df.shape}")

# Select the main patient table for demonstration (using the first available record set)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nFields in main record set '@id': {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    print("\n-- Head of main record set DataFrame --")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We perform basic processing: filter by a numeric field, normalize it, and optionally group by a key field. All fields are referenced by their `@id`.

In [ ]:
# Choose numeric and grouping fields by their @id
# We'll try to infer likely numeric fields from DataFrame
df = dataframes[main_rs_id]
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if not numeric_field:
    # If nothing found, fall back to a known likely numeric field name (manual fallback)
    numeric_field = 'age' if 'age' in df.columns else df.columns[0]

print(f"Using numeric field '@id': {numeric_field}")

# Try to choose a suitable grouping field as well
group_field = None
for col in df.columns:
    if df[col].nunique() < (0.5 * len(df)) and col != numeric_field:
        group_field = col
        break
if not group_field:  # common fallback to a likely field label
    possible_groups = ['sex', 'gender', 'anatomical_location', 'histology', 'msi_status']
    for col in possible_groups:
        if col in df.columns:
            group_field = col
            break

# Filter: for demonstration, threshold on numeric_field > 50 (for eg, age > 50)
threshold = 50
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold}.")
display(filtered_df.head())

# Normalize the numeric field
normcol = f"{numeric_field}_normalized"
filtered_df[normcol] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, normcol]].head())

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} grouped by '{group_field}':")
    display(grouped_df.head())


## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field exists, make a boxplot
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()


## 6. Conclusion
In this notebook, we explored the FAIR² dataset describing clinicopathological and molecular characteristics of second primary colorectal cancer in survivors, loaded it using its Croissant metadata, and performed basic analysis using the `mlcroissant` library.

- We listed all record sets, fields, and demonstrated their usage by referencing their `@id`.
- We extracted tabular data, demonstrated numeric analysis (such as filtering and normalization), grouped results by a selected field, and visualized distributions.
- The flexible Croissant format and the `mlcroissant` Python library make it straightforward to combine FAIR metadata with practical data science analysis.